# quiz 1 基礎 + 進階

In [1]:
import torch
import importlib
import json
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from other_code import quiz1

importlib.reload(quiz1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

/root/venvs/torch-cu121/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda


模型載入(Qwen2.5-3B-Instruct)

In [3]:
model_id = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="cuda:0")

Loading checkpoint shards: 100%|██████████| 2/2 [00:19<00:00,  9.94s/it]


模型測試

In [5]:
model.eval()

messages = [
    {"role": "system", "content": "請使用繁體中文回答。"},
    {"role": "user", "content": "請用三句話介紹什麼是大型語言模型。"},
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=128, do_sample=False, pad_token_id=tokenizer.eos_token_id)

new_tokens = outputs[0, inputs["input_ids"].shape[1]:]
answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

print(answer)

大型語言模型是一種人工智慧技術，它能夠學習和理解大量的文本數據，從而生成合乎語法、邏輯且具有表達力的文本。這些模型通過深度學習方法訓練而成，能夠處理各種語言任務，如文本生成、摘要、對話系統等。


利用20道題目評估LLM

In [5]:
base_dir = Path.cwd()
print("目前工作資料夾：", base_dir)

with open(base_dir / "question/quiz1_baseline.json", "r", encoding="utf-8") as f: questions = json.load(f)
print(f"成功讀取 {len(questions)} 題")

目前工作資料夾： /root/MMIP/week7
成功讀取 20 題


In [ ]:
results = []
all_memory_samples = []
peak_memory_bytes = 0

(base_dir / "result").mkdir(exist_ok=True)

for item in questions:
    print(f"正在回答第{item['id']}題：{item['category']}")

    response, samples, peak = quiz1.answer_with_memory(question=item["question"], model=model, tokenizer=tokenizer)
    all_memory_samples.extend(samples)
    peak_memory_bytes = max(peak_memory_bytes, peak)

    average_mib = sum(samples) / len(samples) / (1024 ** 2)

    results.append({
        "id": item["id"],
        "category": item["category"],
        "question": item["question"],
        "model_answer": response,
        "average_allocated_mib": average_mib,
        "peak_allocated_mib": peak / (1024 ** 2),
    })
    with open(base_dir / "result/quiz1/quiz1_baseline.json", "w", encoding="utf-8",) as f: json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"本題平均顯存：{average_mib:.2f} MiB")
    
summary = {
    "model": model_id,
    "precision": "FP16",
    "prompt_setting": "baseline",
    "measurement": "PyTorch allocated memory",
    "sampling_interval_seconds": 0.05,
    "sample_count": len(all_memory_samples),
    "average_allocated_mib": (sum(all_memory_samples) / len(all_memory_samples) / (1024 ** 2)),
    "peak_allocated_mib": peak_memory_bytes / (1024 ** 2),
}

with open(base_dir / "result/quiz1/quiz1_baseline_memory.json", "w", encoding="utf-8") as f: json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"整體平均顯存：{summary['average_allocated_mib']:.2f} MiB")
print(f"最高顯存：{summary['peak_allocated_mib']:.2f} MiB")    

正在回答第1題：知識理解
本題平均顯存：10432.70 MiB
正在回答第2題：知識理解
本題平均顯存：10440.34 MiB
正在回答第3題：知識理解
本題平均顯存：10439.52 MiB
正在回答第4題：知識理解
本題平均顯存：10433.84 MiB
正在回答第5題：數學與邏輯
本題平均顯存：10433.80 MiB
正在回答第6題：數學與邏輯
本題平均顯存：10434.16 MiB
正在回答第7題：數學與邏輯
本題平均顯存：10434.43 MiB
正在回答第8題：數學與邏輯
本題平均顯存：10437.94 MiB
正在回答第9題：閱讀理解與摘要
本題平均顯存：10434.67 MiB
正在回答第10題：閱讀理解與摘要
本題平均顯存：10436.14 MiB
正在回答第11題：閱讀理解與摘要
本題平均顯存：10435.14 MiB
正在回答第12題：閱讀理解與摘要
本題平均顯存：10434.07 MiB
正在回答第13題：翻譯
本題平均顯存：10431.99 MiB
正在回答第14題：翻譯
本題平均顯存：10432.12 MiB
正在回答第15題：翻譯
本題平均顯存：10432.16 MiB
正在回答第16題：翻譯
本題平均顯存：10432.57 MiB
正在回答第17題：指令與格式遵循
本題平均顯存：10432.62 MiB
正在回答第18題：指令與格式遵循
本題平均顯存：10432.95 MiB
正在回答第19題：指令與格式遵循
本題平均顯存：10432.96 MiB
正在回答第20題：指令與格式遵循
本題平均顯存：10432.58 MiB
整體平均顯存：10437.16 MiB
最高顯存：10538.36 MiB


Prompting Techniques(few shot + prompt用markdown形式)

In [10]:
results = []
all_memory_samples = []
peak_memory_bytes = 0

(base_dir / "result").mkdir(exist_ok=True)

for item in questions:
    print(f"正在回答第{item['id']}題：{item['category']}")

    response, samples, peak = quiz1.answer_with_memory(question=item["question"], model=model, tokenizer=tokenizer, use_prompt_engineering=True)
    all_memory_samples.extend(samples)
    peak_memory_bytes = max(peak_memory_bytes, peak)

    average_mib = sum(samples) / len(samples) / (1024 ** 2)

    results.append({
        "id": item["id"],
        "category": item["category"],
        "question": item["question"],
        "model_answer": response,
        "average_allocated_mib": average_mib,
        "peak_allocated_mib": peak / (1024 ** 2),
    })
    with open(base_dir / "result/quiz1/quiz1_fewshot_markdown.json", "w", encoding="utf-8",) as f: json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"本題平均顯存：{average_mib:.2f} MiB")
    
summary = {
    "model": model_id,
    "precision": "FP16",
    "prompt_setting": "fewshot_markdown",
    "measurement": "PyTorch allocated memory",
    "sampling_interval_seconds": 0.05,
    "sample_count": len(all_memory_samples),
    "average_allocated_mib": (sum(all_memory_samples) / len(all_memory_samples) / (1024 ** 2)),
    "peak_allocated_mib": peak_memory_bytes / (1024 ** 2),
}

with open(base_dir / "result/quiz1/quiz1_fewshot_markdown_memory.json", "w", encoding="utf-8") as f: json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"整體平均顯存：{summary['average_allocated_mib']:.2f} MiB")
print(f"最高顯存：{summary['peak_allocated_mib']:.2f} MiB")    

正在回答第1題：知識理解
本題平均顯存：6601.38 MiB
正在回答第2題：知識理解
本題平均顯存：6609.35 MiB
正在回答第3題：知識理解
本題平均顯存：6607.03 MiB
正在回答第4題：知識理解
本題平均顯存：6601.74 MiB
正在回答第5題：數學與邏輯
本題平均顯存：6601.06 MiB
正在回答第6題：數學與邏輯
本題平均顯存：6601.91 MiB
正在回答第7題：數學與邏輯
本題平均顯存：6602.14 MiB
正在回答第8題：數學與邏輯
本題平均顯存：6605.80 MiB
正在回答第9題：閱讀理解與摘要
本題平均顯存：6605.76 MiB
正在回答第10題：閱讀理解與摘要
本題平均顯存：6605.82 MiB
正在回答第11題：閱讀理解與摘要
本題平均顯存：6606.50 MiB
正在回答第12題：閱讀理解與摘要
本題平均顯存：6605.36 MiB
正在回答第13題：翻譯
本題平均顯存：6601.15 MiB
正在回答第14題：翻譯
本題平均顯存：6601.21 MiB
正在回答第15題：翻譯
本題平均顯存：6601.43 MiB
正在回答第16題：翻譯
本題平均顯存：6602.02 MiB
正在回答第17題：指令與格式遵循
本題平均顯存：6602.22 MiB
正在回答第18題：指令與格式遵循
本題平均顯存：6602.46 MiB
正在回答第19題：指令與格式遵循
本題平均顯存：6603.75 MiB
正在回答第20題：指令與格式遵循
本題平均顯存：6602.85 MiB
整體平均顯存：6606.56 MiB
最高顯存：6918.23 MiB


使用NF4的4-bit權重量化

In [2]:
model_id = "Qwen/Qwen2.5-3B-Instruct"
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=False)
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=quantization_config,
                                             torch_dtype=torch.float16, device_map="cuda:0")
model.eval()

print("是否以4-bit載入：", getattr(model, "is_loaded_in_4bit", False))

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
Loading checkpoint shards: 100%|██████████| 2/2 [00:11<00:00,  5.51s/it]


是否以4-bit載入： True


In [7]:
results = []
all_memory_samples = []
peak_memory_bytes = 0

(base_dir / "result").mkdir(exist_ok=True)

for item in questions:
    print(f"正在回答第{item['id']}題：{item['category']}")

    response, samples, peak = quiz1.answer_with_memory(question=item["question"], model=model, tokenizer=tokenizer)
    all_memory_samples.extend(samples)
    peak_memory_bytes = max(peak_memory_bytes, peak)

    average_mib = sum(samples) / len(samples) / (1024 ** 2)

    results.append({
        "id": item["id"],
        "category": item["category"],
        "question": item["question"],
        "model_answer": response,
        "average_allocated_mib": average_mib,
        "peak_allocated_mib": peak / (1024 ** 2),
    })
    with open(base_dir / "result/quiz1/quiz1_4bit.json", "w", encoding="utf-8",) as f: json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"本題平均顯存：{average_mib:.2f} MiB")
    
summary = {
    "model": model_id,
    "precision": "4-bit NF4 weights; FP16 compute",
    "prompt_setting": "baseline",
    "measurement": "PyTorch allocated memory",
    "sampling_interval_seconds": 0.05,
    "sample_count": len(all_memory_samples),
    "average_allocated_mib": (sum(all_memory_samples) / len(all_memory_samples) / (1024 ** 2)),
    "peak_allocated_mib": peak_memory_bytes / (1024 ** 2),
}

with open(base_dir / "result/quiz1/quiz1_4bit_memory.json", "w", encoding="utf-8") as f: json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"整體平均顯存：{summary['average_allocated_mib']:.2f} MiB")
print(f"最高顯存：{summary['peak_allocated_mib']:.2f} MiB")    

正在回答第1題：知識理解
本題平均顯存：2113.75 MiB
正在回答第2題：知識理解
本題平均顯存：2120.78 MiB
正在回答第3題：知識理解
本題平均顯存：2119.73 MiB
正在回答第4題：知識理解
本題平均顯存：2115.25 MiB
正在回答第5題：數學與邏輯
本題平均顯存：2115.08 MiB
正在回答第6題：數學與邏輯
本題平均顯存：2116.18 MiB
正在回答第7題：數學與邏輯
本題平均顯存：2116.01 MiB
正在回答第8題：數學與邏輯
本題平均顯存：2118.91 MiB
正在回答第9題：閱讀理解與摘要
本題平均顯存：2118.19 MiB
正在回答第10題：閱讀理解與摘要
本題平均顯存：2117.64 MiB
正在回答第11題：閱讀理解與摘要
本題平均顯存：2118.62 MiB
正在回答第12題：閱讀理解與摘要
本題平均顯存：2117.82 MiB
正在回答第13題：翻譯
本題平均顯存：2113.99 MiB
正在回答第14題：翻譯
本題平均顯存：2113.80 MiB
正在回答第15題：翻譯
本題平均顯存：2113.67 MiB
正在回答第16題：翻譯
本題平均顯存：2114.53 MiB
正在回答第17題：指令與格式遵循
本題平均顯存：2114.73 MiB
正在回答第18題：指令與格式遵循
本題平均顯存：2120.63 MiB
正在回答第19題：指令與格式遵循
本題平均顯存：2115.65 MiB
正在回答第20題：指令與格式遵循
本題平均顯存：2117.39 MiB
整體平均顯存：2118.31 MiB
最高顯存：2172.53 MiB


Prompting Techniques(few shot + prompt用markdown形式) + 使用NF4的4-bit權重量化

In [9]:
results = []
all_memory_samples = []
peak_memory_bytes = 0

(base_dir / "result").mkdir(exist_ok=True)

for item in questions:
    print(f"正在回答第{item['id']}題：{item['category']}")

    response, samples, peak = quiz1.answer_with_memory(question=item["question"], model=model, tokenizer=tokenizer)
    all_memory_samples.extend(samples)
    peak_memory_bytes = max(peak_memory_bytes, peak)

    average_mib = sum(samples) / len(samples) / (1024 ** 2)

    results.append({
        "id": item["id"],
        "category": item["category"],
        "question": item["question"],
        "model_answer": response,
        "average_allocated_mib": average_mib,
        "peak_allocated_mib": peak / (1024 ** 2),
    })
    with open(base_dir / "result/quiz1/quiz1_4bit_fewshot_markdown.json", "w", encoding="utf-8",) as f: json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"本題平均顯存：{average_mib:.2f} MiB")
    
summary = {
    "model": model_id,
    "precision": "4-bit NF4 weights; FP16 compute",
    "prompt_setting": "markdown_fewshot",
    "measurement": "PyTorch allocated memory",
    "sampling_interval_seconds": 0.05,
    "sample_count": len(all_memory_samples),
    "average_allocated_mib": (sum(all_memory_samples) / len(all_memory_samples) / (1024 ** 2)),
    "peak_allocated_mib": peak_memory_bytes / (1024 ** 2),
}

with open(base_dir / "result/quiz1/quiz1_4bit_fewshot_markdown_memory.json", "w", encoding="utf-8") as f: json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"整體平均顯存：{summary['average_allocated_mib']:.2f} MiB")
print(f"最高顯存：{summary['peak_allocated_mib']:.2f} MiB")    

正在回答第1題：知識理解
本題平均顯存：2113.75 MiB
正在回答第2題：知識理解
本題平均顯存：2120.56 MiB
正在回答第3題：知識理解
本題平均顯存：2119.62 MiB
正在回答第4題：知識理解
本題平均顯存：2114.80 MiB
正在回答第5題：數學與邏輯
本題平均顯存：2115.06 MiB
正在回答第6題：數學與邏輯
本題平均顯存：2116.59 MiB
正在回答第7題：數學與邏輯
本題平均顯存：2115.88 MiB
正在回答第8題：數學與邏輯
本題平均顯存：2118.68 MiB
正在回答第9題：閱讀理解與摘要
本題平均顯存：2118.35 MiB
正在回答第10題：閱讀理解與摘要
本題平均顯存：2117.81 MiB
正在回答第11題：閱讀理解與摘要
本題平均顯存：2118.54 MiB
正在回答第12題：閱讀理解與摘要
本題平均顯存：2117.79 MiB
正在回答第13題：翻譯
本題平均顯存：2113.80 MiB
正在回答第14題：翻譯
本題平均顯存：2113.85 MiB
正在回答第15題：翻譯
本題平均顯存：2115.95 MiB
正在回答第16題：翻譯
本題平均顯存：2114.38 MiB
正在回答第17題：指令與格式遵循
本題平均顯存：2114.70 MiB
正在回答第18題：指令與格式遵循
本題平均顯存：2120.36 MiB
正在回答第19題：指令與格式遵循
本題平均顯存：2115.90 MiB
正在回答第20題：指令與格式遵循
本題平均顯存：2114.80 MiB
整體平均顯存：2118.30 MiB
最高顯存：2172.53 MiB


# quiz 2 基礎

In [1]:
import torch
import transformers
import importlib
import json
from pathlib import Path
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, Gemma3ForConditionalGeneration
from other_code import quiz2

importlib.reload(quiz2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

/root/venvs/torch-cu121/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda


模型載入(Qwen2.5-VL-3B-Instruct)

In [9]:
model_id = "Qwen/Qwen2.5-VL-3B-Instruct"
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16,
                                                           device_map={"": 0}, attn_implementation="sdpa")
processor = AutoProcessor.from_pretrained(model_id, min_pixels=256 * 28 * 28, max_pixels=1280 * 28 * 28)
model.eval()

Loading checkpoint shards: 100%|██████████| 2/2 [00:09<00:00,  4.77s/it]


Qwen2_5_VLForConditionalGeneration(
  (visual): Qwen2_5_VisionTransformerPretrainedModel(
    (patch_embed): Qwen2_5_VisionPatchEmbed(
      (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
    )
    (rotary_pos_emb): Qwen2_5_VisionRotaryEmbedding()
    (blocks): ModuleList(
      (0-31): 32 x Qwen2_5_VLVisionBlock(
        (norm1): Qwen2RMSNorm((1280,), eps=1e-06)
        (norm2): Qwen2RMSNorm((1280,), eps=1e-06)
        (attn): Qwen2_5_VLVisionSdpaAttention(
          (qkv): Linear(in_features=1280, out_features=3840, bias=True)
          (proj): Linear(in_features=1280, out_features=1280, bias=True)
        )
        (mlp): Qwen2_5_VLMLP(
          (gate_proj): Linear(in_features=1280, out_features=3420, bias=True)
          (up_proj): Linear(in_features=1280, out_features=3420, bias=True)
          (down_proj): Linear(in_features=3420, out_features=1280, bias=True)
          (act_fn): SiLU()
        )
      )
    )
    (merger): Qwen2_5_VLPatchMerger

模型測試

In [ ]:
base_dir = Path.cwd()
print("目前工作目錄：", base_dir)
image_path = (base_dir / "question" / "quiz2_image" / Path(item["image"]).name)

with open(base_dir / "question/quiz2.json", "r", encoding="utf-8") as f: questions = json.load(f)

item = questions[0]

response = quiz2.generate_answer(question=item["question"], image_path=image_path, model=model, processor=processor)

print("問題：", item["question"])
print("模型回答：", response)

目前工作目錄： /root/MMIP/week7


/root/venvs/torch-cu121/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-06` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


問題： 圖片中的動物是什麼？
模型回答： 圖片中的動物是鹿。


利用20道題目評估VLM

In [ ]:
result_dir = base_dir / "result" / "quiz2"
result_dir.mkdir(parents=True, exist_ok=True)

with open(base_dir / "question" / "quiz2.json", "r", encoding="utf-8") as f: questions = json.load(f)

ids = [item["id"] for item in questions]
assert len(ids) == 20 and set(ids) == set(range(1, 21)), \
    "題目應為20題，id須為1～20且不重複"

for item in questions:
    image_path = (base_dir / "question" / "quiz2_image" / Path(item["image"]).name)
    if not image_path.is_file():
        raise FileNotFoundError(f"找不到圖片：{image_path}")

results = []
output_path = result_dir / "quiz2_qwen_baseline.json"

for index, item in enumerate(questions, start=1):
    image_path = (base_dir / "question" / "quiz2_image" / Path(item["image"]).name)
    response = quiz2.generate_answer(question=item["question"], image_path=image_path, model=model, processor=processor)

    results.append({
        "id": item["id"],
        "category": item["category"],
        "image": str(image_path.relative_to(base_dir)),
        "question": item["question"],
        "response": response,
    })
    with open(output_path, "w", encoding="utf-8") as f: json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"[{index}/{len(questions)}]")

print("完成，結果存入：", output_path)

[1/20]
[2/20]
[3/20]
[4/20]
[5/20]
[6/20]
[7/20]
[8/20]
[9/20]
[10/20]
[11/20]
[12/20]
[13/20]
[14/20]
[15/20]
[16/20]
[17/20]
[18/20]
[19/20]
[20/20]
完成，結果存入： /root/MMIP/week7/result/quiz2/quiz2_qwen_baseline.json


模型載入(gemma-3-4b-it)

In [ ]:
from huggingface_hub import login
from getpass import getpass

login(token=getpass("請輸入Hugging Face Read Token："), add_to_git_credential=False)

In [6]:
model_id = "google/gemma-3-4b-it"

model = Gemma3ForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
    attn_implementation="sdpa",
)
model.eval()

Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.72s/it]


Gemma3ForConditionalGeneration(
  (vision_tower): SiglipVisionModel(
    (vision_model): SiglipVisionTransformer(
      (embeddings): SiglipVisionEmbeddings(
        (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
        (position_embedding): Embedding(4096, 1152)
      )
      (encoder): SiglipEncoder(
        (layers): ModuleList(
          (0-26): 27 x SiglipEncoderLayer(
            (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
            (self_attn): SiglipAttention(
              (k_proj): Linear(in_features=1152, out_features=1152, bias=True)
              (v_proj): Linear(in_features=1152, out_features=1152, bias=True)
              (q_proj): Linear(in_features=1152, out_features=1152, bias=True)
              (out_proj): Linear(in_features=1152, out_features=1152, bias=True)
            )
            (layer_norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
            (mlp): SiglipMLP(
            

模型測試

In [7]:
base_dir = Path.cwd()
with open(base_dir / "question/quiz2.json", "r", encoding="utf-8") as f: questions = json.load(f)

item = questions[0]

processor = AutoProcessor.from_pretrained(model_id)
response = quiz2.generate_answer_gemma(question=item["question"], image_path=(base_dir / "question/quiz2_image" / Path(item["image"]).name),
                                       model=model, processor=processor)

print("問題：", item["question"])
print("模型回答：", response)

問題： 圖片中的動物是什麼？
模型回答： 圖片中的動物是鹿。 牠們看起來像是白尾鹿。


利用20道題目評估VLM

In [5]:
result_dir = base_dir / "result" / "quiz2"
result_dir.mkdir(parents=True, exist_ok=True)

with open(base_dir / "question" / "quiz2.json", "r", encoding="utf-8") as f: questions = json.load(f)

ids = [item["id"] for item in questions]
assert len(ids) == 20 and set(ids) == set(range(1, 21)), \
    "題目應為20題，id須為1～20且不重複"

for item in questions:
    image_path = (base_dir / "question" / "quiz2_image" / Path(item["image"]).name)
    if not image_path.is_file():
        raise FileNotFoundError(f"找不到圖片：{image_path}")

results = []
output_path = result_dir / "quiz2_gemma_baseline.json"

for index, item in enumerate(questions, start=1):
    image_path = (base_dir / "question" / "quiz2_image" / Path(item["image"]).name)
    response = quiz2.generate_answer_gemma(question=item["question"], image_path=image_path, model=model, processor=processor)

    results.append({
        "id": item["id"],
        "category": item["category"],
        "image": str(image_path.relative_to(base_dir)),
        "question": item["question"],
        "response": response,
    })
    with open(output_path, "w", encoding="utf-8") as f: json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"[{index}/{len(questions)}]")

print("完成，結果存入：", output_path)

[1/20]
[2/20]
[3/20]
[4/20]
[5/20]
[6/20]
[7/20]
[8/20]
[9/20]
[10/20]
[11/20]
[12/20]
[13/20]
[14/20]
[15/20]
[16/20]
[17/20]
[18/20]
[19/20]
[20/20]
完成，結果存入： /root/MMIP/week7/result/quiz2/quiz2_gemma_baseline.json


# quiz 3 基礎

In [1]:
import torch
import importlib
import json
from pathlib import Path
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed, Qwen2_5_VLForConditionalGeneration, AutoProcessor
from torch.distributed.fsdp import FSDPModule
from trl import DPOConfig, DPOTrainer
from other_code import quiz2, quiz3

importlib.reload(quiz2)
importlib.reload(quiz3)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

/root/venvs/torch-cu121/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda


模型載入(Qwen2.5-3B-Instruct)

In [3]:
model_id = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map={"": 0})
model.eval()

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
Loading checkpoint shards: 100%|██████████| 2/2 [00:54<00:00, 27.43s/it]


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((2048,), eps=1e-06)
    (rotary_emb):

利用20+100(通用題)道題目評估LLM

In [6]:
base_dir = Path.cwd()
result_dir = base_dir / "result" / "quiz3"
result_dir.mkdir(parents=True, exist_ok=True)

model.eval()

test_sets = [
    {
        "input": "quiz3_json_test.json",
        "output": "quiz3_qwen_baseline_json_test.json",
        "count": 20,
    },
    {
        "input": "quiz3_general_benchmark.json",
        "output": "quiz3_qwen_baseline_general.json",
        "count": 100,
    },
]

loaded_sets = []

for setting in test_sets:
    input_path = base_dir / "question" / setting["input"]

    with open(input_path, "r", encoding="utf-8") as f: questions = json.load(f)

    ids = [item["id"] for item in questions]
    assert (len(ids) == setting["count"] and set(ids) == set(range(1, setting["count"] + 1))), f"{setting['input']}：題數或id有問題"

    loaded_sets.append((setting, questions))

for setting, questions in loaded_sets:
    output_path = result_dir / setting["output"]
    results = []

    for index, item in enumerate(questions, start=1):
        response = quiz3.generate_quiz3_answer(question=item["question"], model=model, tokenizer=tokenizer)

        results.append({
            "id": item["id"],
            "category": item["category"],
            "question": item["question"],
            "response": response,
        })

        with open(output_path, "w", encoding="utf-8") as f: json.dump(results, f, ensure_ascii=False, indent=2)

        print(f"[{index}/{len(questions)}]")

    print("已完成：", output_path)

[1/20]
[2/20]
[3/20]
[4/20]
[5/20]
[6/20]
[7/20]
[8/20]
[9/20]
[10/20]
[11/20]
[12/20]
[13/20]
[14/20]
[15/20]
[16/20]
[17/20]
[18/20]
[19/20]
[20/20]
已完成： /root/MMIP/week7/result/quiz3/quiz3_qwen_baseline_json_test.json
[1/100]
[2/100]
[3/100]
[4/100]
[5/100]
[6/100]
[7/100]
[8/100]
[9/100]
[10/100]
[11/100]
[12/100]
[13/100]
[14/100]
[15/100]
[16/100]
[17/100]
[18/100]
[19/100]
[20/100]
[21/100]
[22/100]
[23/100]
[24/100]
[25/100]
[26/100]
[27/100]
[28/100]
[29/100]
[30/100]
[31/100]
[32/100]
[33/100]
[34/100]
[35/100]
[36/100]
[37/100]
[38/100]
[39/100]
[40/100]
[41/100]
[42/100]
[43/100]
[44/100]
[45/100]
[46/100]
[47/100]
[48/100]
[49/100]
[50/100]
[51/100]
[52/100]
[53/100]
[54/100]
[55/100]
[56/100]
[57/100]
[58/100]
[59/100]
[60/100]
[61/100]
[62/100]
[63/100]
[64/100]
[65/100]
[66/100]
[67/100]
[68/100]
[69/100]
[70/100]
[71/100]
[72/100]
[73/100]
[74/100]
[75/100]
[76/100]
[77/100]
[78/100]
[79/100]
[80/100]
[81/100]
[82/100]
[83/100]
[84/100]
[85/100]
[86/100]
[87/100]
[88/1

DPO+LoRA微調

In [2]:
set_seed(42)
base_dir = Path.cwd()

model_id = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = "right"

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map={"": 0}, attn_implementation="sdpa")
model.config.use_cache = False

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
Loading checkpoint shards: 100%|██████████| 2/2 [00:06<00:00,  3.07s/it]


In [3]:
train_dataset = quiz3.load_dpo_dataset(base_dir / "question/quiz3_dpo_train.json")
trainer = quiz3.create_dpo_trainer(model=model, tokenizer=tokenizer, train_dataset=train_dataset, 
                                   output_dir=base_dir / "result/quiz3/dpo_checkpoints")

print("訓練組數：", len(train_dataset))
trainer.model.print_trainable_parameters()

Tokenizing train dataset: 100%|██████████| 100/100 [00:00<00:00, 1297.48 examples/s]
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


訓練組數： 100
trainable params: 1,843,200 || all params: 3,087,781,888 || trainable%: 0.0597


In [4]:
train_result = trainer.train()
adapter_dir = base_dir / "result/quiz3/qwen_dpo_adapter"

trainer.save_model(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
trainer.save_metrics("train", train_result.metrics)
trainer.save_state()

print("訓練完成，儲存位置：", adapter_dir)

Step,Training Loss
1,0.693100
2,0.674400
3,0.646800
4,0.583200
5,0.579000
6,0.512800
7,0.533900
8,0.476000
9,0.458000
10,0.438600


訓練完成，儲存位置： /root/MMIP/week7/result/quiz3/qwen_dpo_adapter


利用20+100(通用題)道題目評估DPO+LoRA微調後的LLM

In [6]:
model = trainer.model
model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()
tokenizer.padding_side = "left"

In [ ]:
base_dir = Path.cwd()
result_dir = base_dir / "result/quiz3"
result_dir.mkdir(parents=True, exist_ok=True)

test_sets = [
    (
        "quiz3_json_test.json",
        "quiz3_qwen_dpo_json_test.json",
        20,
    ),
    (
        "quiz3_general_benchmark.json",
        "quiz3_qwen_dpo_general.json",
        100,
    ),
]

for input_name, output_name, count in test_sets:
    with open(base_dir / "question" / input_name, "r", encoding="utf-8") as f: questions = json.load(f)

    ids = [item["id"] for item in questions]
    assert len(ids) == count and set(ids) == set(range(1, count + 1))

    results = []
    output_path = result_dir / output_name

    print("開始測試：", input_name)

    for index, item in enumerate(questions, start=1):
        response = quiz3.generate_quiz3_answer(question=item["question"], model=model, tokenizer=tokenizer)
        results.append({
            "id": item["id"],
            "category": item["category"],
            "question": item["question"],
            "response": response,
        })

        with open(output_path, "w", encoding="utf-8") as f: json.dump(results, f, ensure_ascii=False, indent=2)
        print(f"[{index}/{count}]")

    print("結果存入：", output_path)

開始測試： quiz3_json_test.json


/root/venvs/torch-cu121/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/root/venvs/torch-cu121/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/root/venvs/torch-cu121/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


[1/20]
[2/20]
[3/20]
[4/20]
[5/20]
[6/20]
[7/20]
[8/20]
[9/20]
[10/20]
[11/20]
[12/20]
[13/20]
[14/20]
[15/20]
[16/20]
[17/20]
[18/20]
[19/20]
[20/20]
結果存入： /root/MMIP/week7/result/quiz3/quiz3_qwen_dpo_json_test.json
開始測試： quiz3_general_benchmark.json
[1/100]
[2/100]
[3/100]
[4/100]
[5/100]
[6/100]
[7/100]
[8/100]
[9/100]
[10/100]
[11/100]
[12/100]
[13/100]
[14/100]
[15/100]
[16/100]
[17/100]
[18/100]
[19/100]
[20/100]
[21/100]
[22/100]
[23/100]
[24/100]
[25/100]
[26/100]
[27/100]
[28/100]
[29/100]
[30/100]
[31/100]
[32/100]
[33/100]
[34/100]
[35/100]
[36/100]
[37/100]
[38/100]
[39/100]
[40/100]
[41/100]
[42/100]
[43/100]
[44/100]
[45/100]
[46/100]
[47/100]
[48/100]
[49/100]
[50/100]
[51/100]
[52/100]
[53/100]
[54/100]
[55/100]
[56/100]
[57/100]
[58/100]
[59/100]
[60/100]
[61/100]
[62/100]
[63/100]
[64/100]
[65/100]
[66/100]
[67/100]
[68/100]
[69/100]
[70/100]
[71/100]
[72/100]
[73/100]
[74/100]
[75/100]
[76/100]
[77/100]
[78/100]
[79/100]
[80/100]
[81/100]
[82/100]
[83/100]
[84/100]
[

利用python評估格式通過率、內容與結構通過率

In [ ]:
base_dir = Path.cwd()
result_dir = base_dir / "result/quiz3"
answer_path = base_dir / "question/quiz3_json_test_answer.json"

comparison = {}

for setting, filename in [
    ("baseline", "quiz3_qwen_baseline_json_test.json"),
    ("dpo", "quiz3_qwen_dpo_json_test.json"),
]:
    scores = quiz3.evaluate_json_test(result_path=result_dir / filename, answer_path=answer_path)

    comparison[setting] = scores

    print(f"設定：{setting}")
    print(
        f"JSON格式通過：{scores['format_pass_count']}/20，"
        f"{scores['format_pass_rate_percent']:.1f}%"
    )
    print(
        f"內容與結構通過：{scores['content_schema_pass_count']}/20，"
        f"{scores['content_schema_pass_rate_percent']:.1f}%"
    )
    print()

with open(result_dir / "quiz3_json_comparison.json", "w", encoding="utf-8") as f: 
    json.dump(comparison, f, ensure_ascii=False, indent=2)

設定：baseline
JSON格式通過：3/20，15.0%
內容與結構通過：3/20，15.0%

設定：dpo
JSON格式通過：20/20，100.0%
內容與結構通過：16/20，80.0%



# quiz 3 進階

透過Hugging Face下載RLAIF-V-Dataset，取100組並保存圖片與JSON

In [2]:
base_dir = Path.cwd()

image_dir = base_dir / "question/quiz3_train_images"
image_dir.mkdir(parents=True, exist_ok=True)

dataset = load_dataset("openbmb/RLAIF-V-Dataset", split="train", streaming=True)

results = []
seen = set()

for source_index, item in enumerate(dataset):
    question = item["question"].strip()
    chosen = item["chosen"].strip()
    rejected = item["rejected"].strip()

    key = (question, chosen, rejected)

    if (
        not question
        or not chosen
        or not rejected
        or chosen == rejected
        or key in seen
    ):
        continue

    image = item["image"]
    if image is None:
        continue

    sample_id = len(results) + 1
    image_path = image_dir / f"{sample_id:03d}.jpg"

    image.convert("RGB").save(image_path, quality=95)

    results.append({
        "id": sample_id,
        "image": str(image_path.relative_to(base_dir)),
        "prompt": question,
        "chosen": chosen,
        "rejected": rejected,
        "source": "openbmb/RLAIF-V-Dataset",
        "source_index": source_index,
        "source_id": str(item.get("idx", "")),
    })

    seen.add(key)

    if len(results) == 100:
        break

assert len(results) == 100, "未取得完整100組資料"

output_path = base_dir / "question/quiz3_vl_dpo_train.json"

with open(output_path, "w", encoding="utf-8") as f: json.dump(results, f, ensure_ascii=False, indent=2)

print("已保存偏好資料：", len(results), "組")
print("JSON位置：", output_path)
print("圖片位置：", image_dir)

已保存偏好資料： 100 組
JSON位置： /root/MMIP/week7/question/quiz3_vl_dpo_train.json
圖片位置： /root/MMIP/week7/question/quiz3_train_images


調整./question/quiz2_image 20組prompt的圖片亮度、畫質

In [ ]:
base_dir = Path.cwd()
output_dir = quiz3.create_test_images(source_dir=base_dir / "question/quiz2_image", output_dir=base_dir / "question/quiz3_test_images")

print("圖片變化完成，存放於：", output_dir)

圖片變化完成，存放於： /root/MMIP/week7/question/quiz3_test_images


模型載入(Qwen2.5-VL-3B-Instruct)

In [ ]:
model_id = "Qwen/Qwen2.5-VL-3B-Instruct"
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16,
                                                           device_map={"": 0}, attn_implementation="sdpa")
processor = AutoProcessor.from_pretrained(model_id, min_pixels=256 * 28 * 28, max_pixels=1280 * 28 * 28)
model.eval()

Loading checkpoint shards: 100%|██████████| 2/2 [01:06<00:00, 33.48s/it]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Qwen2_5_VLForConditionalGeneration(
  (visual): Qwen2_5_VisionTransformerPretrainedModel(
    (patch_embed): Qwen2_5_VisionPatchEmbed(
      (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
    )
    (rotary_pos_emb): Qwen2_5_VisionRotaryEmbedding()
    (blocks): ModuleList(
      (0-31): 32 x Qwen2_5_VLVisionBlock(
        (norm1): Qwen2RMSNorm((1280,), eps=1e-06)
        (norm2): Qwen2RMSNorm((1280,), eps=1e-06)
        (attn): Qwen2_5_VLVisionSdpaAttention(
          (qkv): Linear(in_features=1280, out_features=3840, bias=True)
          (proj): Linear(in_features=1280, out_features=1280, bias=True)
        )
        (mlp): Qwen2_5_VLMLP(
          (gate_proj): Linear(in_features=1280, out_features=3420, bias=True)
          (up_proj): Linear(in_features=1280, out_features=3420, bias=True)
          (down_proj): Linear(in_features=3420, out_features=1280, bias=True)
          (act_fn): SiLU()
        )
      )
    )
    (merger): Qwen2_5_VLPatchMerger

利用三種調整過圖片的prompt各20道題目評估VLM

In [ ]:
result_dir = base_dir / "result" / "quiz3"
result_dir.mkdir(parents=True, exist_ok=True)

with open(base_dir / "question" / "quiz2.json", "r", encoding="utf-8") as f: questions = json.load(f)

ids = [item["id"] for item in questions]
assert len(ids) == 20 and set(ids) == set(range(1, 21)), \
    "題目應為20題，id須為1～20且不重複"

versions = [
    ("dark", ".png"),
    ("bright", ".png"),
    ("low_quality", ".jpg"),
]

image_root = base_dir / "question/quiz3_test_images"

for version, extension in versions:
    for item in questions:
        image_name = Path(item["image"]).stem + extension
        image_path = image_root / version / image_name

        if not image_path.is_file():
            raise FileNotFoundError(f"找不到圖片：{image_path}")

for version, extension in versions:
    results = []
    output_path = result_dir / f"quiz3_qwen_vl_baseline_{version}.json"

    print("開始測試：", version)

    for index, item in enumerate(questions, start=1):
        image_name = Path(item["image"]).stem + extension
        image_path = image_root / version / image_name

        response = quiz2.generate_answer(question=item["question"], image_path=image_path, model=model, processor=processor)

        results.append({
            "id": item["id"],
            "category": item["category"],
            "version": version,
            "image": str(image_path.relative_to(base_dir)),
            "question": item["question"],
            "response": response,
        })

        with open(output_path, "w", encoding="utf-8") as f: json.dump(results, f, ensure_ascii=False, indent=2)
        print(f"[{version}] [{index}/{len(questions)}]")

    print("完成，結果存入：", output_path)

開始測試： dark


/root/venvs/torch-cu121/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-06` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


[dark] [1/20]
[dark] [2/20]
[dark] [3/20]
[dark] [4/20]
[dark] [5/20]
[dark] [6/20]
[dark] [7/20]
[dark] [8/20]
[dark] [9/20]
[dark] [10/20]
[dark] [11/20]
[dark] [12/20]
[dark] [13/20]
[dark] [14/20]
[dark] [15/20]
[dark] [16/20]
[dark] [17/20]
[dark] [18/20]
[dark] [19/20]
[dark] [20/20]
完成，結果存入： /root/MMIP/week7/result/quiz3/quiz3_qwen_vl_baseline_dark.json
開始測試： bright
[bright] [1/20]
[bright] [2/20]
[bright] [3/20]
[bright] [4/20]
[bright] [5/20]
[bright] [6/20]
[bright] [7/20]
[bright] [8/20]
[bright] [9/20]
[bright] [10/20]
[bright] [11/20]
[bright] [12/20]
[bright] [13/20]
[bright] [14/20]
[bright] [15/20]
[bright] [16/20]
[bright] [17/20]
[bright] [18/20]
[bright] [19/20]
[bright] [20/20]
完成，結果存入： /root/MMIP/week7/result/quiz3/quiz3_qwen_vl_baseline_bright.json
開始測試： low_quality
[low_quality] [1/20]
[low_quality] [2/20]
[low_quality] [3/20]
[low_quality] [4/20]
[low_quality] [5/20]
[low_quality] [6/20]
[low_quality] [7/20]
[low_quality] [8/20]
[low_quality] [9/20]
[low_quality

DPO＋LoRA微調

In [2]:
set_seed(42)
base_dir = Path.cwd()
model_id = "Qwen/Qwen2.5-VL-3B-Instruct"

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16, device_map={"": 0}, attn_implementation="sdpa")
processor = AutoProcessor.from_pretrained(model_id,  min_pixels=256 * 28 * 28, max_pixels=1280 * 28 * 28)
processor.tokenizer.padding_side = "left"

if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

model.config.use_cache = False

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:06<00:00,  3.16s/it]
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


In [3]:
train_dataset = quiz3.load_vlm_dpo_dataset(file_path=base_dir / "question/quiz3_vl_dpo_train.json", base_dir=base_dir)
trainer = quiz3.create_vlm_dpo_trainer(model=model, processor=processor, train_dataset=train_dataset, 
                                       output_dir=base_dir / "result/quiz3/vl_dpo_checkpoints")

print("訓練組數：", len(train_dataset))
trainer.model.print_trainable_parameters()

The model is already on multiple devices. Skipping the move to device specified in `args`.


訓練組數： 100
trainable params: 1,843,200 || all params: 3,756,466,176 || trainable%: 0.0491


In [4]:
from types import MethodType

def skip_model_card(self, *args, **kwargs):
    return None

trainer.create_model_card = MethodType(skip_model_card, trainer)

In [6]:
train_result = trainer.train()
adapter_dir = base_dir / "result/quiz3/qwen_vl_dpo_adapter"

trainer.save_model(str(adapter_dir))
processor.save_pretrained(str(adapter_dir))
trainer.save_metrics("train", train_result.metrics)
trainer.save_state()

print("訓練完成，Adapter存入：", adapter_dir)

Step,Training Loss
1,0.693100
2,0.692700
3,0.693100
4,0.691100
5,0.693100
6,0.691500
7,0.687500
8,0.690600
9,0.606600
10,0.689800


訓練完成，Adapter存入： /root/MMIP/week7/result/quiz3/qwen_vl_dpo_adapter


利用baseline+三種調整過圖片的prompt各20道題目評估DPO+LoRA微調後VLM

In [7]:
model = trainer.model

model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()

processor.tokenizer.padding_side = "left"

In [8]:
result_dir = base_dir / "result/quiz3"
result_dir.mkdir(parents=True, exist_ok=True)

with open(base_dir / "question/quiz2.json", "r", encoding="utf-8") as f: questions = json.load(f)

ids = [item["id"] for item in questions]
assert len(ids) == 20 and set(ids) == set(range(1, 21))

image_root = base_dir / "question/quiz3_test_images"

versions = [
    ("original", base_dir / "question/quiz2_image", ".jpg"),
    ("dark", image_root / "dark", ".png"),
    ("bright", image_root / "bright", ".png"),
    ("low_quality", image_root / "low_quality", ".jpg"),
]

for version, image_dir, extension in versions:
    for item in questions:
        image_path = image_dir / (Path(item["image"]).stem + extension)
        if not image_path.is_file():
            raise FileNotFoundError(f"找不到圖片：{image_path}")

for version, image_dir, extension in versions:
    results = []
    output_path = (result_dir / f"quiz3_qwen_vl_dpo_{version}.json")

    for index, item in enumerate(questions, start=1):
        image_path = image_dir / (Path(item["image"]).stem + extension)

        response = quiz2.generate_answer(question=item["question"], image_path=image_path, model=model, processor=processor)
        results.append({
            "id": item["id"],
            "category": item["category"],
            "version": version,
            "image": str(image_path.relative_to(base_dir)),
            "question": item["question"],
            "response": response,
        })

        with open(output_path, "w", encoding="utf-8") as f: json.dump(results, f, ensure_ascii=False, indent=2)

        print(f"[{version}] [{index}/{len(questions)}]")

    print("完成，結果存入：", output_path)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[original] [1/20]
[original] [2/20]
[original] [3/20]
[original] [4/20]
[original] [5/20]
[original] [6/20]
[original] [7/20]
[original] [8/20]
[original] [9/20]
[original] [10/20]
[original] [11/20]
[original] [12/20]
[original] [13/20]
[original] [14/20]
[original] [15/20]
[original] [16/20]
[original] [17/20]
[original] [18/20]
[original] [19/20]
[original] [20/20]
完成，結果存入： /root/MMIP/week7/result/quiz3/quiz3_qwen_vl_dpo_original.json
[dark] [1/20]
[dark] [2/20]
[dark] [3/20]
[dark] [4/20]
[dark] [5/20]
[dark] [6/20]
[dark] [7/20]
[dark] [8/20]
[dark] [9/20]
[dark] [10/20]
[dark] [11/20]
[dark] [12/20]
[dark] [13/20]
[dark] [14/20]
[dark] [15/20]
[dark] [16/20]
[dark] [17/20]
[dark] [18/20]
[dark] [19/20]
[dark] [20/20]
完成，結果存入： /root/MMIP/week7/result/quiz3/quiz3_qwen_vl_dpo_dark.json
[bright] [1/20]
[bright] [2/20]
[bright] [3/20]
[bright] [4/20]
[bright] [5/20]
[bright] [6/20]
[bright] [7/20]
[bright] [8/20]
[bright] [9/20]
[bright] [10/20]
[bright] [11/20]
[bright] [12/20]
[brig